In [ ]:
# ARC-AGI-3 submission - GraphExplorer (CPU, sin GPU)
import os, subprocess, sys, time
from pathlib import Path

NOTEBOOK_START = time.time()
TRUE_SUBMISSION = os.environ.get("KAGGLE_IS_COMPETITION_RERUN", "").strip().lower() in {"1", "true"}
os.environ["ONLY_RESET_LEVELS"] = "true"   # RESET reinicia el nivel, no el juego
print("TRUE_SUBMISSION =", TRUE_SUBMISSION)

COMP_ROOT = None
for dirpath, dirnames, _ in os.walk("/kaggle/input"):
    if "arc_agi_3_wheels" in dirnames:
        COMP_ROOT = Path(dirpath)
        break
assert COMP_ROOT is not None, "wheelhouse de la competencia no encontrado"
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "--no-index",
                       "--no-warn-conflicts", "--disable-pip-version-check",
                       f"--find-links={COMP_ROOT / 'arc_agi_3_wheels'}", "arc-agi"],
                      stdout=subprocess.DEVNULL)
import arc_agi
print("arc_agi OK, COMP_ROOT =", COMP_ROOT)

# Kaggle exige un submission.parquet; el score real sale de las partidas contra el gateway.
import pandas as pd
pd.DataFrame([["1_0", "1", True, 1]],
             columns=["row_id", "game_id", "end_of_game", "score"]).to_parquet(
    "/kaggle/working/submission.parquet", index=False)


In [ ]:
# Reconstruye src/arc3 embebido (generado por build_submit_notebook.py)
import os
os.makedirs('/kaggle/working/src/arc3', exist_ok=True)
SOURCES = {
 "src/arc3/__init__.py": "\"\"\"arc3: utilidades para ARC-AGI-3 (Kaggle arc-prize-2026-arc-agi-3).\n\nM\u00f3dulos:\n  env      -> descubrimiento y ejecuci\u00f3n local de environments (arcengine/arc_agi)\n  features -> feature engineering sobre frames 64x64 y transiciones (s, a, s')\n  probe    -> pol\u00edtica de sondeo que genera el dataset de features por juego\n\"\"\"\n\nfrom .features import (\n    connected_components,\n    grid_features,\n    frame_to_grid,\n    transition_features,\n)\n\n__all__ = [\n    \"connected_components\",\n    \"grid_features\",\n    \"frame_to_grid\",\n    \"transition_features\",\n]\n",
 "src/arc3/agent.py": "\"\"\"GraphExplorer: agente de exploraci\u00f3n de grafo de estados para ARC-AGI-3.\n\nS\u00edntesis de lo mejor del leaderboard p\u00fablico (ver docs/STRATEGY.md):\n  - Grafo de estados con hashing enmascarado (borde 3px + m\u00e1scara de contador aprendida),\n    BFS sobre el grafo aprendido para volver a nodos con acciones pendientes, y replay\n    tras RESET aprovechando el determinismo de los juegos.  [estilo v47, LB 0.54]\n  - Clicks por componentes conexas ordenadas por button-likeness (compacto+peque\u00f1o+color\n    raro) + rejilla gruesa de cobertura; supresi\u00f3n \"deadsig\" de clases estructuralmente\n    inertes con protecci\u00f3n de clases alguna vez efectivas.   [estilo 2\u00ba milestone]\n  - Orden de acciones simples por P(cambio) aprendida online; no-ops se hunden, no se podan.\n\nL\u00f3gica pura sobre numpy: el runner (local o gateway) le pasa frames y ejecuta lo que elige.\n\"\"\"\n\nfrom __future__ import annotations\n\nfrom collections import deque\nfrom typing import Any, Optional\n\nimport numpy as np\n\nfrom .features import connected_components\n\nGRID = 64\nBORDER = 3           # borde enmascarado del hash: HUD/contadores viven ah\u00ed\nCOUNTER_WARMUP = 12  # transiciones para aprender la m\u00e1scara de contador\nCOUNTER_FRACTION = 0.8   # celda contador si cambia en >=80% de las transiciones\nCOUNTER_MAX_INTERIOR = 0.2  # la m\u00e1scara aprendida no puede tapar >20% del interior\nCLICK_CAP = 64       # candidatos de click por nodo\nDEAD_K = 2           # clase de click muerta tras K usos inertes\nMAX_EXHAUSTED_RESETS = 40   # reinicios diversificados antes de rendirse\nRESET_LOOP_BREAK = 30\n\n# ids de acci\u00f3n: 0=RESET, 1..5 y 7 simples, 6=click(x,y)\nSIMPLE_IDS = (1, 2, 3, 4, 5, 7)\nRESET_KEY = (0, -1, -1)\n\n\nclass _Node:\n    __slots__ = (\"pending\", \"tried\")\n\n    def __init__(self) -> None:\n        self.pending: deque[tuple[int, int, int]] = deque()\n        self.tried: set[tuple[int, int, int]] = set()\n\n\nclass GraphExplorer:\n    \"\"\"Elige (action_id, x, y). El caller ejecuta y devuelve el frame en el pr\u00f3ximo choose().\"\"\"\n\n    def __init__(self, game_id: str = \"\", max_actions: int = 15000) -> None:\n        self.game_id = game_id\n        self.max_actions = max_actions\n        self.actions_taken = 0\n\n        self._nodes: dict[int, _Node] = {}\n        self._edges: dict[tuple[int, tuple[int, int, int]], int] = {}\n        self._adj: dict[int, list[tuple[tuple[int, int, int], int]]] = {}\n\n        self._counter_counts = np.zeros((GRID, GRID), dtype=np.int32)\n        self._counter_seen = 0\n        self._counter_mask: Optional[np.ndarray] = None\n\n        # stats por acci\u00f3n simple: [cambios, usos, nodos_nuevos]\n        self._act_stats: dict[int, list[int]] = {a: [0, 0, 0] for a in SIMPLE_IDS}\n        # deadsig por clase estructural de click (color, size, is_rect)\n        self._dead_sigs: dict[tuple[int, int, bool], int] = {}\n        self._eff_sigs: set[tuple[int, int, bool]] = set()\n\n        self._last_key: Optional[int] = None\n        self._last_action: Optional[tuple[int, int, int]] = None\n        self._last_grid: Optional[np.ndarray] = None\n        self._last_levels = 0\n        self._replay: deque[tuple[int, int, int]] = deque()\n        self._replay_target: Optional[int] = None\n        self._exhausted_resets = 0\n        self._consecutive_resets = 0\n        self._salt = 0   # perturba el orden de candidatos en cada reinicio diversificado\n        self.done = False\n\n    # ---------- hashing ----------\n\n    def _mask(self) -> np.ndarray:\n        m = np.zeros((GRID, GRID), dtype=bool)\n        m[:BORDER, :] = m[-BORDER:, :] = m[:, :BORDER] = m[:, -BORDER:] = True\n        if self._counter_mask is not None:\n            m |= self._counter_mask\n        return m\n\n    def _key(self, grid: np.ndarray) -> int:\n        g = grid.copy()\n        g[self._mask()] = 0\n        return hash(g.tobytes())\n\n    def _learn_counter_mask(self, prev: np.ndarray, nxt: np.ndarray) -> None:\n        if self._counter_mask is not None or prev is None:\n            return\n        self._counter_counts += prev != nxt\n        self._counter_seen += 1\n        if self._counter_seen >= COUNTER_WARMUP:\n            cand = self._counter_counts >= COUNTER_FRACTION * self._counter_seen\n            cand[:BORDER, :] = cand[-BORDER:, :] = cand[:, :BORDER] = cand[:, -BORDER:] = False\n            interior = (GRID - 2 * BORDER) ** 2\n            # si \"todo cambia siempre\" (animaci\u00f3n global) la m\u00e1scara ser\u00eda in\u00fatil: borde solo\n            self._counter_mask = cand if cand.sum() <= COUNTER_MAX_INTERIOR * interior \\\n                else np.zeros((GRID, GRID), dtype=bool)\n\n    # ---------- candidatos ----------\n\n    def _click_sig(self, obj: dict[str, Any]) -> tuple[int, int, bool]:\n        y0, x0, y1, x1 = obj[\"bbox\"]\n        is_rect = obj[\"size\"] == (y1 - y0 + 1) * (x1 - x0 + 1)\n        return (obj[\"color\"], obj[\"size\"], is_rect)\n\n    def _click_candidates(self, grid: np.ndarray) -> list[tuple[int, int, int]]:\n        counts = np.bincount(grid.ravel(), minlength=16)\n        background = int(counts.argmax())\n        total = grid.size\n        objs = connected_components(grid, background)\n        scored = []\n        for o in objs:\n            sig = self._click_sig(o)\n            if self._dead_sigs.get(sig, 0) >= DEAD_K and sig not in self._eff_sigs:\n                continue\n            rarity = 1.0 - counts[o[\"color\"]] / total\n            y0, x0, y1, x1 = o[\"bbox\"]\n            fill = o[\"size\"] / ((y1 - y0 + 1) * (x1 - x0 + 1))\n            size_score = 1.0 if o[\"size\"] <= 4 else 0.8 if o[\"size\"] <= 16 else \\\n                0.5 if o[\"size\"] <= 64 else 0.25 if o[\"size\"] <= 256 else 0.0\n            score = 0.4 * rarity + 0.3 * size_score + 0.3 * fill\n            cy, cx = o[\"centroid\"]\n            scored.append((score, int(round(cx)), int(round(cy))))\n        scored.sort(key=lambda t: -t[0])\n        cands = [(6, x, y) for _, x, y in scored[:CLICK_CAP]]\n        # rejilla de cobertura; se densifica con cada reinicio diversificado (salt)\n        stride = 8 if self._salt == 0 else 4 if self._salt < 3 else 2\n        offset = (self._salt * 3) % max(stride, 1)\n        cap = CLICK_CAP if self._salt == 0 else CLICK_CAP * 4\n        for gy in range(offset, GRID, stride):\n            for gx in range(offset, GRID, stride):\n                if len(cands) >= cap:\n                    break\n                c = (6, gx, gy)\n                if c not in cands:\n                    cands.append(c)\n        return cands[:cap]\n\n    def _simple_order(self, available: list[int]) -> list[int]:\n        acts = [a for a in SIMPLE_IDS if not available or a in available]\n        # rotaci\u00f3n por salt: cada reinicio diversificado prueba un orden base distinto,\n        # as\u00ed el desempate entre acciones no probadas genera trayectorias nuevas.\n        if self._salt and acts:\n            r = self._salt % len(acts)\n            acts = acts[r:] + acts[:r]\n\n        def score(a: int) -> float:\n            chg, uses, _new = self._act_stats[a]\n            return 0.5 if uses == 0 else chg / uses\n\n        return sorted(acts, key=lambda a: -score(a))\n\n    def _fill_pending(self, node: _Node, grid: np.ndarray, available: list[int]) -> None:\n        for a in self._simple_order(available):\n            k = (a, -1, -1)\n            if k not in node.tried:\n                node.pending.append(k)\n        if not available or 6 in available:\n            for k in self._click_candidates(grid):\n                if k not in node.tried:\n                    node.pending.append(k)\n\n    # ---------- grafo ----------\n\n    def _record_edge(self, src: int, action: tuple[int, int, int], dst: int) -> None:\n        if (src, action) not in self._edges:\n            self._edges[(src, action)] = dst\n            self._adj.setdefault(src, []).append((action, dst))\n\n    def _bfs_to_pending(self, start: int) -> Optional[list[tuple[int, int, int]]]:\n        \"\"\"Camino m\u00e1s corto (en aristas conocidas) hasta un nodo con pendientes.\"\"\"\n        seen = {start}\n        q: deque[tuple[int, list[tuple[int, int, int]]]] = deque([(start, [])])\n        while q:\n            key, path = q.popleft()\n            node = self._nodes.get(key)\n            if node and node.pending and key != start:\n                return path\n            if len(path) >= 60:\n                continue\n            for action, dst in self._adj.get(key, []):\n                if dst not in seen:\n                    seen.add(dst)\n                    q.append((dst, path + [action]))\n        return None\n\n    # ---------- API ----------\n\n    def choose(\n        self,\n        grid: np.ndarray,\n        state: str,\n        levels_completed: int,\n        available_actions: list[int],\n    ) -> tuple[int, int, int]:\n        \"\"\"Devuelve (action_id, x, y); x=y=-1 para acciones simples/RESET.\"\"\"\n        self.actions_taken += 1\n        if self.actions_taken > self.max_actions:\n            self.done = True\n\n        # --- digerir el resultado de la acci\u00f3n anterior ---\n        if self._last_grid is not None and self._last_action is not None:\n            changed = bool((self._last_grid != grid).any())\n            self._learn_counter_mask(self._last_grid, grid)\n            aid, ax, ay = self._last_action\n            if aid in self._act_stats:\n                self._act_stats[aid][1] += 1\n                self._act_stats[aid][0] += int(changed)\n                self._act_stats[aid][2] += int(self._key(grid) not in self._nodes)\n            if aid == 6:\n                sig = self._sig_at(self._last_grid, ax, ay)\n                if sig is not None:\n                    if changed or levels_completed != self._last_levels:\n                        self._eff_sigs.add(sig)\n                    else:\n                        self._dead_sigs[sig] = self._dead_sigs.get(sig, 0) + 1\n\n        if levels_completed > self._last_levels:\n            # nivel nuevo: lo inerte de antes puede ser la clave ahora\n            self._dead_sigs.clear()\n            self._eff_sigs.clear()\n            self._replay.clear()\n            self._replay_target = None\n        self._last_levels = levels_completed\n\n        key = self._key(grid)\n        if self._last_key is not None and self._last_action is not None:\n            self._record_edge(self._last_key, self._last_action, key)\n\n        # --- game over / not played ---\n        if state in (\"NOT_PLAYED\", \"GAME_OVER\"):\n            self._consecutive_resets += 1\n            if self._consecutive_resets >= RESET_LOOP_BREAK:\n                self.done = True\n            return self._commit(grid, key, RESET_KEY)\n        self._consecutive_resets = 0\n\n        # --- replay en curso (verificando determinismo) ---\n        if self._replay:\n            if self._replay_target is not None and key != self._replay_target:\n                self._replay.clear()  # el mundo no sigui\u00f3 el grafo: abortar replay\n                self._replay_target = None\n            else:\n                action = self._replay.popleft()\n                self._replay_target = self._edges.get((key, action))\n                return self._commit(grid, key, action)\n\n        node = self._nodes.get(key)\n        if node is None:\n            node = _Node()\n            self._nodes[key] = node\n            self._fill_pending(node, grid, available_actions)\n\n        if node.pending:\n            action = node.pending.popleft()\n            node.tried.add(action)\n            return self._commit(grid, key, action)\n\n        # nodo agotado: BFS al nodo pendiente m\u00e1s cercano\n        path = self._bfs_to_pending(key)\n        if path:\n            self._replay = deque(path)\n            action = self._replay.popleft()\n            self._replay_target = self._edges.get((key, action))\n            return self._commit(grid, key, action)\n\n        # grafo alcanzable agotado: reinicio DIVERSIFICADO. En juegos deterministas,\n        # re-explorar con el mismo orden repetir\u00eda la trayectoria; subimos el salt para\n        # densificar clicks y perturbar el orden, y abrimos de nuevo la exploraci\u00f3n\n        # (olvidamos pending/tried; conservamos dead/eff sigs y la m\u00e1scara aprendida).\n        self._exhausted_resets += 1\n        self._salt += 1\n        self._nodes.clear()\n        self._edges.clear()\n        self._adj.clear()\n        self._replay.clear()\n        self._replay_target = None\n        if self._exhausted_resets >= MAX_EXHAUSTED_RESETS:\n            self.done = True\n        return self._commit(grid, key, RESET_KEY)\n\n    def _sig_at(self, grid: np.ndarray, x: int, y: int) -> Optional[tuple[int, int, bool]]:\n        counts = np.bincount(grid.ravel(), minlength=16)\n        background = int(counts.argmax())\n        if not (0 <= x < GRID and 0 <= y < GRID) or grid[y, x] == background:\n            return None\n        for o in connected_components(grid, background):\n            y0, x0, y1, x1 = o[\"bbox\"]\n            if y0 <= y <= y1 and x0 <= x <= x1:\n                return self._click_sig(o)\n        return None\n\n    def _commit(self, grid: np.ndarray, key: int, action: tuple[int, int, int]) -> tuple[int, int, int]:\n        self._last_grid = grid.copy()\n        self._last_key = key\n        self._last_action = action\n        return action\n",
 "src/arc3/env.py": "\"\"\"Descubrimiento y ejecuci\u00f3n local de environments ARC-AGI-3.\n\nEnvuelve arc_agi.LocalEnvironmentWrapper para jugar los juegos de\nenvironment_files/ sin API remota (igual que har\u00e1 el rerun de Kaggle offline).\n\"\"\"\n\nfrom __future__ import annotations\n\nimport json\nimport logging\nfrom pathlib import Path\nfrom typing import Any, Optional\n\nfrom arc_agi.local_wrapper import LocalEnvironmentWrapper\nfrom arc_agi.models import EnvironmentInfo\nfrom arcengine import FrameDataRaw, GameAction\n\nlogger = logging.getLogger(\"arc3\")\n\n\ndef discover_environments(env_root: Path) -> list[EnvironmentInfo]:\n    \"\"\"Lista los environments locales a partir de environment_files/<game>/<hash>/metadata.json.\"\"\"\n    infos: list[EnvironmentInfo] = []\n    for meta_path in sorted(env_root.glob(\"*/*/metadata.json\")):\n        meta = json.loads(meta_path.read_text(encoding=\"utf-8\"))\n        # local_dir del metadata es relativo al root del dataset; usamos el real.\n        meta[\"local_dir\"] = str(meta_path.parent)\n        infos.append(EnvironmentInfo.model_validate(meta))\n    return infos\n\n\nclass LocalGame:\n    \"\"\"Sesi\u00f3n de un juego local: reset/step con FrameDataRaw.\"\"\"\n\n    def __init__(self, info: EnvironmentInfo, seed: int = 0) -> None:\n        self.info = info\n        self.env = LocalEnvironmentWrapper(\n            environment_info=info,\n            logger=logger,\n            scorecard_id=\"local-probe\",\n            seed=seed,\n            save_recording=False,\n        )\n\n    def reset(self) -> Optional[FrameDataRaw]:\n        return self.env.reset()\n\n    def step(\n        self, action: GameAction, x: Optional[int] = None, y: Optional[int] = None\n    ) -> Optional[FrameDataRaw]:\n        data: dict[str, Any] = {\"game_id\": self.info.game_id}\n        if action.is_complex():\n            data[\"x\"] = int(x or 0)\n            data[\"y\"] = int(y or 0)\n        return self.env.step(action, data=data)\n",
 "src/arc3/features.py": "\"\"\"Feature engineering para frames de ARC-AGI-3.\n\nLos frames son grids 64x64 con colores 0..15. Aqu\u00ed se computan:\n  - features por frame (histograma de color, objetos, simetr\u00edas, bordes, entrop\u00eda)\n  - features de transici\u00f3n (s, a, s'): p\u00edxeles cambiados, bbox del cambio,\n    deltas por color y detecci\u00f3n de traslaci\u00f3n (vector de movimiento)\n\nTodo en numpy puro (sin scipy) para poder correr offline en Kaggle sin deps extra.\n\"\"\"\n\nfrom __future__ import annotations\n\nfrom collections import deque\nfrom typing import Any, Optional, Sequence\n\nimport numpy as np\n\nN_COLORS = 16\nGRID = 64\n# Desplazamientos m\u00e1ximos a testear al detectar traslaci\u00f3n de objetos entre frames.\nMAX_SHIFT = 8\n\n\ndef frame_to_grid(frame: Any) -> np.ndarray:\n    \"\"\"Convierte FrameData.frame (lista de grids; puede traer varios por animaci\u00f3n)\n    al \u00faltimo grid como np.ndarray (64, 64) int8.\"\"\"\n    if frame is None or len(frame) == 0:\n        return np.zeros((GRID, GRID), dtype=np.int8)\n    last = frame[-1]\n    return np.asarray(last, dtype=np.int8)\n\n\ndef connected_components(\n    grid: np.ndarray, background: Optional[int] = None\n) -> list[dict[str, Any]]:\n    \"\"\"Componentes conexas 4-conectadas de celdas del mismo color (ignora el fondo).\n\n    Devuelve una lista de objetos: color, size, bbox (y0, x0, y1, x1), centroid.\n    BFS puro en python: el grid es 64x64, es barato.\n    \"\"\"\n    h, w = grid.shape\n    if background is None:\n        background = int(np.bincount(grid.ravel(), minlength=N_COLORS).argmax())\n    seen = np.zeros((h, w), dtype=bool)\n    objects: list[dict[str, Any]] = []\n    for y in range(h):\n        for x in range(w):\n            if seen[y, x] or grid[y, x] == background:\n                continue\n            color = int(grid[y, x])\n            q = deque([(y, x)])\n            seen[y, x] = True\n            cells = []\n            while q:\n                cy, cx = q.popleft()\n                cells.append((cy, cx))\n                for ny, nx in ((cy - 1, cx), (cy + 1, cx), (cy, cx - 1), (cy, cx + 1)):\n                    if 0 <= ny < h and 0 <= nx < w and not seen[ny, nx] and grid[ny, nx] == color:\n                        seen[ny, nx] = True\n                        q.append((ny, nx))\n            ys = [c[0] for c in cells]\n            xs = [c[1] for c in cells]\n            objects.append(\n                {\n                    \"color\": color,\n                    \"size\": len(cells),\n                    \"bbox\": (min(ys), min(xs), max(ys), max(xs)),\n                    \"centroid\": (float(np.mean(ys)), float(np.mean(xs))),\n                }\n            )\n    objects.sort(key=lambda o: -o[\"size\"])\n    return objects\n\n\ndef _edge_density(grid: np.ndarray) -> float:\n    \"\"\"Fracci\u00f3n de pares vecinos (4-conn) con colores distintos: mide 'estructura'.\"\"\"\n    dh = grid[:, 1:] != grid[:, :-1]\n    dv = grid[1:, :] != grid[:-1, :]\n    return float((dh.sum() + dv.sum()) / (dh.size + dv.size))\n\n\ndef _entropy(counts: np.ndarray) -> float:\n    p = counts[counts > 0].astype(np.float64)\n    p /= p.sum()\n    return float(-(p * np.log2(p)).sum())\n\n\ndef grid_features(grid: np.ndarray, max_objects: int = 8) -> dict[str, Any]:\n    \"\"\"Features escalares de un grid 64x64.\"\"\"\n    counts = np.bincount(grid.ravel(), minlength=N_COLORS)[:N_COLORS]\n    background = int(counts.argmax())\n    objects = connected_components(grid, background)\n    feats: dict[str, Any] = {\n        \"background\": background,\n        \"n_colors\": int((counts > 0).sum()),\n        \"color_entropy\": _entropy(counts),\n        \"edge_density\": _edge_density(grid),\n        \"sym_h\": float((grid == grid[:, ::-1]).mean()),  # simetr\u00eda izquierda-derecha\n        \"sym_v\": float((grid == grid[::-1, :]).mean()),  # simetr\u00eda arriba-abajo\n        \"n_objects\": len(objects),\n    }\n    for c in range(N_COLORS):\n        feats[f\"color_{c}\"] = int(counts[c])\n    for i in range(max_objects):\n        if i < len(objects):\n            o = objects[i]\n            y0, x0, y1, x1 = o[\"bbox\"]\n            feats[f\"obj{i}_color\"] = o[\"color\"]\n            feats[f\"obj{i}_size\"] = o[\"size\"]\n            feats[f\"obj{i}_cy\"], feats[f\"obj{i}_cx\"] = o[\"centroid\"]\n            feats[f\"obj{i}_h\"], feats[f\"obj{i}_w\"] = y1 - y0 + 1, x1 - x0 + 1\n        else:\n            feats[f\"obj{i}_color\"] = -1\n            feats[f\"obj{i}_size\"] = 0\n            feats[f\"obj{i}_cy\"] = feats[f\"obj{i}_cx\"] = -1.0\n            feats[f\"obj{i}_h\"] = feats[f\"obj{i}_w\"] = 0\n    return feats\n\n\ndef _detect_translation(prev: np.ndarray, nxt: np.ndarray, diff: np.ndarray) -> tuple[int, int, float]:\n    \"\"\"Busca el shift (dy, dx) que mejor explica el cambio como traslaci\u00f3n.\n\n    Solo mira la regi\u00f3n cambiada: si nxt == shift(prev) sobre esa regi\u00f3n, hay\n    movimiento de un objeto. Devuelve (dy, dx, score) con score en [0, 1].\n    \"\"\"\n    ys, xs = np.nonzero(diff)\n    if len(ys) == 0:\n        return 0, 0, 0.0\n    best = (0, 0, 0.0)\n    for dy in range(-MAX_SHIFT, MAX_SHIFT + 1):\n        for dx in range(-MAX_SHIFT, MAX_SHIFT + 1):\n            if dy == 0 and dx == 0:\n                continue\n            sy, sx = ys - dy, xs - dx\n            ok = (sy >= 0) & (sy < GRID) & (sx >= 0) & (sx < GRID)\n            if not ok.any():\n                continue\n            match = float((nxt[ys[ok], xs[ok]] == prev[sy[ok], sx[ok]]).mean())\n            if match > best[2]:\n                best = (dy, dx, match)\n    return best\n\n\ndef transition_features(prev_grid: np.ndarray, next_grid: np.ndarray) -> dict[str, Any]:\n    \"\"\"Features del cambio entre dos frames consecutivos.\"\"\"\n    diff = prev_grid != next_grid\n    n_changed = int(diff.sum())\n    feats: dict[str, Any] = {\"n_changed\": n_changed}\n    if n_changed == 0:\n        feats.update(\n            {\"chg_y0\": -1, \"chg_x0\": -1, \"chg_h\": 0, \"chg_w\": 0,\n             \"chg_area_frac\": 0.0, \"move_dy\": 0, \"move_dx\": 0, \"move_score\": 0.0,\n             \"colors_gained\": 0, \"colors_lost\": 0}\n        )\n        return feats\n    ys, xs = np.nonzero(diff)\n    y0, y1, x0, x1 = ys.min(), ys.max(), xs.min(), xs.max()\n    feats[\"chg_y0\"], feats[\"chg_x0\"] = int(y0), int(x0)\n    feats[\"chg_h\"], feats[\"chg_w\"] = int(y1 - y0 + 1), int(x1 - x0 + 1)\n    feats[\"chg_area_frac\"] = float(n_changed / diff.size)\n    prev_counts = np.bincount(prev_grid.ravel(), minlength=N_COLORS)[:N_COLORS]\n    next_counts = np.bincount(next_grid.ravel(), minlength=N_COLORS)[:N_COLORS]\n    delta = next_counts.astype(int) - prev_counts.astype(int)\n    feats[\"colors_gained\"] = int((delta > 0).sum())\n    feats[\"colors_lost\"] = int((delta < 0).sum())\n    dy, dx, score = _detect_translation(prev_grid, next_grid, diff)\n    feats[\"move_dy\"], feats[\"move_dx\"], feats[\"move_score\"] = dy, dx, score\n    return feats\n\n\ndef action_effect_summary(rows: Sequence[dict[str, Any]]) -> list[dict[str, Any]]:\n    \"\"\"Resumen por acci\u00f3n a partir de filas de transici\u00f3n: \u00bfqu\u00e9 acciones 'hacen algo'?\n\n    Cada fila debe traer: action_id, n_changed, level_up (bool), game_over (bool).\n    \"\"\"\n    out: list[dict[str, Any]] = []\n    by_action: dict[int, list[dict[str, Any]]] = {}\n    for r in rows:\n        by_action.setdefault(int(r[\"action_id\"]), []).append(r)\n    for action_id, rs in sorted(by_action.items()):\n        n = len(rs)\n        out.append(\n            {\n                \"action_id\": action_id,\n                \"n_uses\": n,\n                \"p_change\": float(np.mean([r[\"n_changed\"] > 0 for r in rs])),\n                \"avg_pixels_changed\": float(np.mean([r[\"n_changed\"] for r in rs])),\n                \"p_level_up\": float(np.mean([bool(r.get(\"level_up\")) for r in rs])),\n                \"p_game_over\": float(np.mean([bool(r.get(\"game_over\")) for r in rs])),\n            }\n        )\n    return out\n",
 "src/arc3/probe.py": "\"\"\"Pol\u00edtica de sondeo: juega cada environment y produce el dataset de features.\n\nEstrategia por juego:\n  1. RESET y features del frame inicial.\n  2. Round-robin sobre las acciones simples disponibles (ACTION1..5, 7) para\n     perfilar qu\u00e9 hace cada una (\u00bfcambia el frame?, \u00bfmueve un objeto?, \u00bfsube nivel?).\n  3. Sondeo de ACTION6 (click x,y) sobre una malla gruesa de puntos, para mapear\n     regiones interactivas.\n  4. Si el juego llega a GAME_OVER se hace RESET y se contin\u00faa hasta agotar budget.\n\nCada paso emite una fila con features de transici\u00f3n + features del frame resultante.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport random\nimport time\nfrom typing import Any, Optional\n\nfrom arcengine import FrameDataRaw, GameAction, GameState\n\nfrom .env import LocalGame\nfrom .features import frame_to_grid, grid_features, transition_features\n\nSIMPLE_ACTIONS = [\n    GameAction.ACTION1,\n    GameAction.ACTION2,\n    GameAction.ACTION3,\n    GameAction.ACTION4,\n    GameAction.ACTION5,\n    GameAction.ACTION7,\n]\n\n\ndef _click_grid(n: int = 8) -> list[tuple[int, int]]:\n    \"\"\"Malla n x n de puntos (x, y) centrados en tiles de 64/n.\"\"\"\n    step = 64 // n\n    half = step // 2\n    return [(x * step + half, y * step + half) for y in range(n) for x in range(n)]\n\n\ndef probe_game(\n    game: LocalGame,\n    budget: int = 300,\n    click_grid_n: int = 8,\n    seed: int = 0,\n    time_limit_s: Optional[float] = None,\n) -> list[dict[str, Any]]:\n    \"\"\"Sondea un juego y devuelve filas de features (una por acci\u00f3n ejecutada).\"\"\"\n    rng = random.Random(seed)\n    rows: list[dict[str, Any]] = []\n    t0 = time.time()\n\n    frame = game.reset()\n    if frame is None:\n        return rows\n    prev_grid = frame_to_grid(frame.frame)\n    prev_levels = frame.levels_completed\n\n    clicks = _click_grid(click_grid_n)\n    rng.shuffle(clicks)\n    click_i = 0\n    step_i = 0\n\n    while step_i < budget:\n        if time_limit_s is not None and time.time() - t0 > time_limit_s:\n            break\n        avail = frame.available_actions or []\n        simple = [a for a in SIMPLE_ACTIONS if not avail or a.value in avail]\n        use_click = (GameAction.ACTION6.value in avail or not avail) and (\n            not simple or step_i % 3 == 2\n        )\n\n        x = y = None\n        if use_click and click_i < len(clicks):\n            action = GameAction.ACTION6\n            x, y = clicks[click_i]\n            click_i += 1\n        elif simple:\n            action = simple[step_i % len(simple)]\n        elif GameAction.ACTION6.value in avail:\n            action = GameAction.ACTION6\n            x, y = rng.randrange(64), rng.randrange(64)\n        else:\n            break\n\n        nxt = game.step(action, x=x, y=y)\n        step_i += 1\n        if nxt is None:\n            continue\n\n        next_grid = frame_to_grid(nxt.frame)\n        row: dict[str, Any] = {\n            \"game_id\": game.info.game_id,\n            \"step\": step_i,\n            \"action_id\": action.value,\n            \"click_x\": -1 if x is None else x,\n            \"click_y\": -1 if y is None else y,\n            \"state\": nxt.state.value,\n            \"levels_completed\": nxt.levels_completed,\n            \"win_levels\": nxt.win_levels,\n            \"level_up\": nxt.levels_completed > prev_levels,\n            \"game_over\": nxt.state == GameState.GAME_OVER,\n            \"win\": nxt.state == GameState.WIN,\n        }\n        row.update(transition_features(prev_grid, next_grid))\n        row.update({f\"nf_{k}\": v for k, v in grid_features(next_grid).items()})\n        rows.append(row)\n\n        prev_levels = nxt.levels_completed\n        prev_grid = next_grid\n        frame = nxt\n\n        if nxt.state in (GameState.GAME_OVER, GameState.WIN):\n            frame = game.reset() or frame\n            prev_grid = frame_to_grid(frame.frame)\n            prev_levels = frame.levels_completed\n\n    return rows\n",
 "src/arc3/runner.py": "\"\"\"Runner paralelo de juegos ARC-AGI-3 sobre un Arcade (offline o gateway).\n\nEn el rerun real cada acci\u00f3n es un request HTTP al gateway (latencia-bound): jugar\nN juegos en paralelo multiplica el throughput de acciones (el milestone winner usaba\nconcurrencia 28). Offline es CPU-bound: pocos workers bastan.\n\nUso (notebook de submission y eval local):\n    arcade = Arcade(operation_mode=..., ...)\n    results = run_games(arcade, game_ids, total_budget_s=..., workers=12)\n\"\"\"\n\nfrom __future__ import annotations\n\nimport threading\nimport time\nfrom typing import Any, Optional\n\nfrom arcengine import GameAction\n\nfrom .agent import GraphExplorer\nfrom .features import frame_to_grid\n\n\ndef play_game(\n    env: Any,\n    game_id: str,\n    time_budget_s: float,\n    max_actions: int = 15000,\n    stop_event: Optional[threading.Event] = None,\n) -> dict[str, Any]:\n    \"\"\"Juega un env (EnvironmentWrapper de arc_agi) con GraphExplorer hasta agotar budget.\"\"\"\n    agent = GraphExplorer(game_id, max_actions=max_actions)\n    t0 = time.time()\n    try:\n        frame = env.observation_space or env.reset()\n    except Exception:\n        frame = None\n    best = 0\n    win = False\n    while (\n        frame is not None\n        and not agent.done\n        and time.time() - t0 < time_budget_s\n        and not (stop_event and stop_event.is_set())\n    ):\n        try:\n            grid = frame_to_grid(frame.frame)\n            aid, x, y = agent.choose(\n                grid, frame.state.value, frame.levels_completed,\n                list(frame.available_actions or []),\n            )\n            action = GameAction.from_id(aid)\n            data: dict[str, Any] = {\"game_id\": game_id}\n            if aid == 6:\n                data.update(x=x, y=y)\n            frame = env.reset() if aid == 0 else env.step(action, data=data)\n        except Exception:\n            try:\n                frame = env.reset()\n            except Exception:\n                break\n        if frame is not None:\n            best = max(best, frame.levels_completed)\n            if frame.state.value == \"WIN\":\n                win = True\n                break\n    return {\n        \"game_id\": game_id,\n        \"levels\": best,\n        \"win\": win,\n        \"actions\": agent.actions_taken,\n        \"seconds\": round(time.time() - t0, 1),\n        \"nodes\": len(agent._nodes),\n    }\n\n\ndef run_games(\n    arcade: Any,\n    game_ids: list[str],\n    total_budget_s: float,\n    workers: int = 8,\n    max_actions: int = 15000,\n    max_game_s: float = 1800.0,\n    min_game_s: float = 60.0,\n    card_id: Optional[str] = None,\n    verbose: bool = True,\n) -> list[dict[str, Any]]:\n    \"\"\"Juega todos los game_ids con un pool de threads y presupuesto global compartido.\"\"\"\n    t_start = time.time()\n    results: list[dict[str, Any]] = []\n    queue = list(game_ids)\n    lock = threading.Lock()\n    stop_event = threading.Event()\n\n    def remaining() -> float:\n        return total_budget_s - (time.time() - t_start)\n\n    def worker() -> None:\n        while not stop_event.is_set():\n            with lock:\n                if not queue:\n                    return\n                games_left = len(queue)\n                game_id = queue.pop(0)\n            rem = remaining()\n            if rem < min_game_s:\n                stop_event.set()\n                return\n            # presupuesto por juego: reparte el tiempo restante entre los juegos que\n            # quedan, multiplicado por los workers (corren en paralelo)\n            budget = max(min_game_s, min(max_game_s, rem * workers / max(games_left, 1)))\n            budget = min(budget, rem)\n            try:\n                with lock:\n                    env = arcade.make(game_id, scorecard_id=card_id)\n                if env is None:\n                    raise RuntimeError(\"make() devolvi\u00f3 None\")\n                r = play_game(env, game_id, budget, max_actions, stop_event)\n            except Exception as e:\n                r = {\"game_id\": game_id, \"levels\": 0, \"win\": False, \"actions\": 0,\n                     \"seconds\": 0.0, \"nodes\": 0, \"error\": str(e)[:200]}\n            with lock:\n                results.append(r)\n                if verbose:\n                    print(f\"[{len(results)}/{len(game_ids)}] {r['game_id']}: \"\n                          f\"{r['levels']} niveles, {r['actions']} acciones, \"\n                          f\"{r['seconds']}s{' WIN' if r.get('win') else ''}\"\n                          f\"{' ERROR ' + r['error'] if r.get('error') else ''}\",\n                          flush=True)\n\n    threads = [threading.Thread(target=worker, daemon=True) for _ in range(workers)]\n    for t in threads:\n        t.start()\n    for t in threads:\n        t.join(timeout=max(0.0, total_budget_s - (time.time() - t_start)) + max_game_s)\n    if verbose:\n        total = sum(r[\"levels\"] for r in results)\n        print(f\"TOTAL: {total} niveles en {len(results)} juegos \"\n              f\"({time.time() - t_start:.0f}s)\", flush=True)\n    return results\n"
}
for path, code in SOURCES.items():
    with open('/kaggle/working/' + path, 'w', encoding='utf-8') as f:
        f.write(code)
print('src/arc3 reconstruido')


In [ ]:
sys.path.insert(0, "/kaggle/working/src")
from urllib.request import urlopen

from arc_agi.base import Arcade, OperationMode
from arc3.runner import run_games

# En rerun cada acción es un request HTTP al gateway (latencia-bound): la concurrencia
# multiplica el throughput (el milestone winner usaba 28). Offline es CPU-bound.
WORKERS = 14 if TRUE_SUBMISSION else 4
TOTAL_BUDGET_S = (8 * 3600 - 900) - (time.time() - NOTEBOOK_START) if TRUE_SUBMISSION else 1200
MAX_ACTIONS_PER_GAME = 15000
MAX_GAME_S = 2400.0 if TRUE_SUBMISSION else 90.0


def wait_for_gateway(base_url, timeout_s=600.0):
    deadline = time.monotonic() + timeout_s
    last = ""
    while time.monotonic() < deadline:
        try:
            with urlopen(f"{base_url}api/games", timeout=10) as r:
                if r.status < 500:
                    return
        except Exception as e:
            last = repr(e)
        time.sleep(5)
    raise RuntimeError(f"gateway no respondió: {last}")


if TRUE_SUBMISSION:
    base_url = os.environ.get("ARC_BASE_URL", "http://gateway:8001/")
    wait_for_gateway(base_url)
    arcade = Arcade(arc_api_key=os.environ.get("ARC_API_KEY", "test-key-123"),
                    arc_base_url=base_url,
                    operation_mode=OperationMode.COMPETITION,
                    environments_dir="")
else:
    arcade = Arcade(operation_mode=OperationMode.OFFLINE,
                    environments_dir=str(COMP_ROOT / "environment_files"))

game_ids = [e.game_id for e in arcade.available_environments]
print(len(game_ids), "juegos,", WORKERS, "workers, budget", int(TOTAL_BUDGET_S), "s")

try:
    card_id = arcade.open_scorecard(tags=["agent", "graph-explorer"])
except Exception as e:
    print("open_scorecard:", e)
    card_id = None

results = run_games(arcade, game_ids, total_budget_s=TOTAL_BUDGET_S, workers=WORKERS,
                    max_actions=MAX_ACTIONS_PER_GAME, max_game_s=MAX_GAME_S,
                    card_id=card_id)


In [ ]:
try:
    sc = arcade.close_scorecard(card_id) if card_id else None
    if sc is not None:
        print(sc.model_dump_json(indent=2)[:4000])
except Exception as e:
    print("close_scorecard:", e)

import pandas as pd
df = pd.DataFrame(results).sort_values("game_id")
df.to_csv("/kaggle/working/results.csv", index=False)
print(df.to_string(index=False))
print("TOTAL:", df.levels.sum(), "niveles en", len(df), "juegos")
